# Complex Test Models for the Explainability App

This notebook trains **deliberately complex** tree-based models on real, open-source
datasets and saves them as `.joblib` files into the `models/` folder so they can be used
to stress-test the ML Explainability Agent.

The app currently understands these model families (see `ingestion/adapters/`):

- **scikit-learn single trees** – `DecisionTreeClassifier`, `DecisionTreeRegressor`
- **scikit-learn ensembles** – `RandomForest*`, `ExtraTrees*`
- **XGBoost** – `XGBClassifier`, `XGBRegressor`

For every model, the app extracts feature names from the estimator's `feature_names_in_`
attribute. To guarantee those are populated, **every model here is fit on a pandas
`DataFrame`** (never a raw NumPy array).

### What this notebook produces

| Dataset | Task | Models saved | Complexity |
|---|---|---|---|
| Adult Census Income | Binary classification | Random Forest, XGBoost | 14 mixed features, deep trees |
| California Housing | Regression | Decision Tree, Random Forest | Deep regression trees |
| Breast Cancer Wisconsin | Binary classification | Extra Trees, Decision Tree | 30 numeric features |

Each dataset also gets a **data dictionary** CSV written under `data/<dataset>/` in the
same schema the app's `DataDictionaryLoader` understands
(`Column, Meaning, Data Type, Unit, Allowed Range`).


In [1]:
# Imports and path setup
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml, fetch_california_housing, load_breast_cancer
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, r2_score
from xgboost import XGBClassifier

# Resolve the project root from the notebook location (notebooks/ -> repo root).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR = ROOT / "models"
DATA_DIR = ROOT / "data"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print(f"Project root: {ROOT}")
print(f"Models dir:   {MODELS_DIR}")
print(f"Data dir:     {DATA_DIR}")


Project root: d:\Projects\Agentic_AI\ml-explainability-agent
Models dir:   d:\Projects\Agentic_AI\ml-explainability-agent\models
Data dir:     d:\Projects\Agentic_AI\ml-explainability-agent\data


## Helper functions

`save_data_dictionary` writes a CSV in the schema the app's `DataDictionaryLoader`
recognises. `save_model` persists a fitted estimator and prints a short summary so we can
confirm the feature names travelled with the model.


In [11]:
def save_data_dictionary(dataset_name: str, rows: list[dict]) -> Path:
    """Write a data dictionary CSV under data/<dataset_name>/data_dictionary.csv.

    Args:
        dataset_name: Folder name under data/ to write into.
        rows: One dict per feature with keys Column, Meaning, Data Type,
            Unit, Allowed Range, Median.

    Returns:
        The path to the written CSV.
    """
    columns = ["Column", "Meaning", "Data Type", "Unit", "Allowed Range", "Median"]
    out_dir = DATA_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "data_dictionary.csv"
    pd.DataFrame(rows, columns=columns).to_csv(out_path, index=False)
    print(f"Data dictionary -> {out_path} ({len(rows)} features)")
    return out_path


def save_model(model, filename: str) -> Path:
    """Persist a fitted estimator to the models/ folder and print a summary."""
    out_path = MODELS_DIR / filename
    joblib.dump(model, out_path)
    feature_names = getattr(model, "feature_names_in_", None)
    n_features = len(feature_names) if feature_names is not None else "?"
    print(f"Model -> {out_path}")
    print(f"    type={type(model).__name__}  n_features={n_features}")
    if feature_names is not None:
        preview = list(feature_names)[:5]
        print(f"    feature_names_in_ (first 5): {preview}")
    return out_path


## Dataset 1 — Adult Census Income (binary classification)

The **Adult** dataset (a.k.a. "Census Income") predicts whether a person earns more than
\$50K/year from 14 demographic and employment features. It mixes numeric and categorical
columns, which makes it a good stress test.

Categorical columns are **ordinal-encoded** to integers so the trees produce interpretable
numeric split thresholds. The original column names are preserved on the resulting
`DataFrame`, so `feature_names_in_` stays human-readable in the saved model.


In [3]:
# Load and preprocess the Adult Census Income dataset (downloaded from OpenML)
adult = fetch_openml("adult", version=2, as_frame=True)
adult_df = adult.frame.copy()

# Drop rows with missing values and split off the target.
adult_df = adult_df.replace("?", np.nan).dropna().reset_index(drop=True)
adult_target_col = "class"
adult_y = (adult_df[adult_target_col].astype(str).str.contains(">50K")).astype(int)
adult_X = adult_df.drop(columns=[adult_target_col])

# Identify categorical vs numeric columns.
adult_cat_cols = adult_X.select_dtypes(include=["category", "object"]).columns.tolist()
adult_num_cols = [c for c in adult_X.columns if c not in adult_cat_cols]

# Ordinal-encode categoricals in place, preserving column names.
adult_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
adult_X_enc = adult_X.copy()
adult_X_enc[adult_cat_cols] = adult_encoder.fit_transform(
    adult_X[adult_cat_cols].astype(str)
)
adult_X_enc = adult_X_enc.astype(float)

print(f"Adult: {adult_X_enc.shape[0]} rows, {adult_X_enc.shape[1]} features")
print(f"Categorical: {adult_cat_cols}")
print(f"Numeric:     {adult_num_cols}")
adult_X_enc.head()


Adult: 45222 rows, 14 features
Categorical: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
Numeric:     ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25.0,2.0,226802.0,1.0,7.0,4.0,6.0,3.0,2.0,1.0,0.0,0.0,40.0,38.0
1,38.0,2.0,89814.0,11.0,9.0,2.0,4.0,0.0,4.0,1.0,0.0,0.0,50.0,38.0
2,28.0,1.0,336951.0,7.0,12.0,2.0,10.0,0.0,4.0,1.0,0.0,0.0,40.0,38.0
3,44.0,2.0,160323.0,15.0,10.0,2.0,6.0,0.0,2.0,1.0,7688.0,0.0,40.0,38.0
4,34.0,2.0,198693.0,0.0,6.0,4.0,7.0,1.0,4.0,1.0,0.0,0.0,30.0,38.0


In [4]:
# Train a deep Random Forest and an XGBoost classifier on Adult, then save both.
X_tr, X_te, y_tr, y_te = train_test_split(
    adult_X_enc, adult_y, test_size=0.2, random_state=RANDOM_STATE, stratify=adult_y
)

adult_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
adult_rf.fit(X_tr, y_tr)
print(f"Adult RandomForest accuracy: {accuracy_score(y_te, adult_rf.predict(X_te)):.4f}")

adult_xgb = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
adult_xgb.fit(X_tr, y_tr)
print(f"Adult XGBoost accuracy:      {accuracy_score(y_te, adult_xgb.predict(X_te)):.4f}")

save_model(adult_rf, "adult_income_random_forest.joblib")
save_model(adult_xgb, "adult_income_xgboost.joblib")


Adult RandomForest accuracy: 0.8595
Adult XGBoost accuracy:      0.8671
Model -> d:\Projects\Agentic_AI\ml-explainability-agent\models\adult_income_random_forest.joblib
    type=RandomForestClassifier  n_features=14
    feature_names_in_ (first 5): ['age', 'workclass', 'fnlwgt', 'education', 'education-num']
Model -> d:\Projects\Agentic_AI\ml-explainability-agent\models\adult_income_xgboost.joblib
    type=XGBClassifier  n_features=14
    feature_names_in_ (first 5): [np.str_('age'), np.str_('workclass'), np.str_('fnlwgt'), np.str_('education'), np.str_('education-num')]


WindowsPath('d:/Projects/Agentic_AI/ml-explainability-agent/models/adult_income_xgboost.joblib')

In [12]:
# Write the Adult data dictionary. For ordinal-encoded categoricals, the "Allowed Range"
# lists the integer codes and their original categories so decision paths stay readable.
# The "Median" column carries a representative value used to pre-fill the UI sample inputs.
adult_meanings = {
    "age": "Age of the individual in years",
    "workclass": "Employment type (private, self-employed, government, etc.)",
    "fnlwgt": "Census final sampling weight for the record",
    "education": "Highest level of education attained",
    "education-num": "Highest education level encoded as an ordinal number",
    "marital-status": "Marital status category",
    "occupation": "Occupation category",
    "relationship": "Relationship role within the household",
    "race": "Self-reported race category",
    "sex": "Self-reported sex",
    "capital-gain": "Capital gains recorded for the year",
    "capital-loss": "Capital losses recorded for the year",
    "hours-per-week": "Hours worked per week",
    "native-country": "Country of origin",
}

adult_rows = []
for col in adult_X_enc.columns:
    col_vals = adult_X_enc[col]
    if col in adult_cat_cols:
        idx = adult_cat_cols.index(col)
        categories = list(adult_encoder.categories_[idx])
        code_map = "; ".join(f"{i}={c}" for i, c in enumerate(categories))
        allowed = f"0-{len(categories) - 1} ({code_map})"
        dtype, unit = "categorical (ordinal-encoded)", "code"
        median_value = round(float(col_vals.median()))
    else:
        allowed = f"{col_vals.min():.0f}-{col_vals.max():.0f}"
        dtype, unit = "numeric", ""
        median_value = round(float(col_vals.median()), 2)
    adult_rows.append({
        "Column": col,
        "Meaning": adult_meanings.get(col, col.replace("-", " ").title()),
        "Data Type": dtype,
        "Unit": unit,
        "Allowed Range": allowed,
        "Median": median_value,
    })

save_data_dictionary("adult_income", adult_rows)


Data dictionary -> d:\Projects\Agentic_AI\ml-explainability-agent\data\adult_income\data_dictionary.csv (14 features)


WindowsPath('d:/Projects/Agentic_AI/ml-explainability-agent/data/adult_income/data_dictionary.csv')

## Dataset 2 — California Housing (regression)

The **California Housing** dataset predicts the median house value (in \$100,000s) for
California districts from 8 numeric features. It exercises the app's **regression** tree
paths and feature importances with a deep single tree and a random forest.


In [6]:
# Load California Housing and train a deep regression tree + random forest.
cal = fetch_california_housing(as_frame=True)
cal_X = cal.data.copy()
cal_y = cal.target.copy()

print(f"California Housing: {cal_X.shape[0]} rows, {cal_X.shape[1]} features")

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    cal_X, cal_y, test_size=0.2, random_state=RANDOM_STATE
)

cal_tree = DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=RANDOM_STATE)
cal_tree.fit(Xc_tr, yc_tr)
print(f"California DecisionTree R^2: {r2_score(yc_te, cal_tree.predict(Xc_te)):.4f}")

cal_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=16,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
cal_rf.fit(Xc_tr, yc_tr)
print(f"California RandomForest R^2: {r2_score(yc_te, cal_rf.predict(Xc_te)):.4f}")

save_model(cal_tree, "california_housing_decision_tree.joblib")
save_model(cal_rf, "california_housing_random_forest.joblib")


California Housing: 20640 rows, 8 features
California DecisionTree R^2: 0.7229
California RandomForest R^2: 0.7991
Model -> d:\Projects\Agentic_AI\ml-explainability-agent\models\california_housing_decision_tree.joblib
    type=DecisionTreeRegressor  n_features=8
    feature_names_in_ (first 5): ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population']
Model -> d:\Projects\Agentic_AI\ml-explainability-agent\models\california_housing_random_forest.joblib
    type=RandomForestRegressor  n_features=8
    feature_names_in_ (first 5): ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population']


WindowsPath('d:/Projects/Agentic_AI/ml-explainability-agent/models/california_housing_random_forest.joblib')

In [13]:
# Write the California Housing data dictionary.
cal_specs = {
    "MedInc": ("Median income in the block group", "tens of thousands USD", "0-15"),
    "HouseAge": ("Median house age in the block group", "years", "1-52"),
    "AveRooms": ("Average number of rooms per household", "rooms", "1-140"),
    "AveBedrms": ("Average number of bedrooms per household", "rooms", "0-35"),
    "Population": ("Block group population", "people", "3-35000"),
    "AveOccup": ("Average number of household members", "people", "1-1250"),
    "Latitude": ("Block group latitude", "degrees", "32-42"),
    "Longitude": ("Block group longitude", "degrees", "-125--114"),
}

cal_rows = [
    {
        "Column": col,
        "Meaning": meaning,
        "Data Type": "numeric",
        "Unit": unit,
        "Allowed Range": allowed,
        "Median": round(float(cal_X[col].median()), 4),
    }
    for col, (meaning, unit, allowed) in cal_specs.items()
]

save_data_dictionary("california_housing", cal_rows)


Data dictionary -> d:\Projects\Agentic_AI\ml-explainability-agent\data\california_housing\data_dictionary.csv (8 features)


WindowsPath('d:/Projects/Agentic_AI/ml-explainability-agent/data/california_housing/data_dictionary.csv')

## Dataset 3 — Breast Cancer Wisconsin (binary classification)

The **Breast Cancer Wisconsin** dataset predicts whether a tumour is malignant or benign
from 30 numeric features derived from cell-nucleus images. Its wide, all-numeric feature
space is a good test for the **Extra Trees** ensemble adapter and for deep single-tree
decision paths.


In [8]:
# Load Breast Cancer and train an Extra Trees ensemble + a deep single tree.
bc = load_breast_cancer(as_frame=True)
bc_X = bc.data.copy()
bc_y = bc.target.copy()  # 0 = malignant, 1 = benign

print(f"Breast Cancer: {bc_X.shape[0]} rows, {bc_X.shape[1]} features")

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    bc_X, bc_y, test_size=0.2, random_state=RANDOM_STATE, stratify=bc_y
)

bc_et = ExtraTreesClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
bc_et.fit(Xb_tr, yb_tr)
print(f"Breast Cancer ExtraTrees accuracy:   {accuracy_score(yb_te, bc_et.predict(Xb_te)):.4f}")

bc_tree = DecisionTreeClassifier(max_depth=8, min_samples_leaf=3, random_state=RANDOM_STATE)
bc_tree.fit(Xb_tr, yb_tr)
print(f"Breast Cancer DecisionTree accuracy: {accuracy_score(yb_te, bc_tree.predict(Xb_te)):.4f}")

save_model(bc_et, "breast_cancer_extra_trees.joblib")
save_model(bc_tree, "breast_cancer_decision_tree.joblib")


Breast Cancer: 569 rows, 30 features
Breast Cancer ExtraTrees accuracy:   0.9474
Breast Cancer DecisionTree accuracy: 0.9123
Model -> d:\Projects\Agentic_AI\ml-explainability-agent\models\breast_cancer_extra_trees.joblib
    type=ExtraTreesClassifier  n_features=30
    feature_names_in_ (first 5): ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']
Model -> d:\Projects\Agentic_AI\ml-explainability-agent\models\breast_cancer_decision_tree.joblib
    type=DecisionTreeClassifier  n_features=30
    feature_names_in_ (first 5): ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']


WindowsPath('d:/Projects/Agentic_AI/ml-explainability-agent/models/breast_cancer_decision_tree.joblib')

In [14]:
# Write the Breast Cancer data dictionary directly from the feature names.
# Features follow the pattern "<measurement> <statistic>" (mean / error / worst).
bc_rows = []
for col in bc_X.columns:
    col_vals = bc_X[col]
    bc_rows.append({
        "Column": col,
        "Meaning": f"{col.capitalize()} of the cell-nucleus measurements",
        "Data Type": "numeric",
        "Unit": "",
        "Allowed Range": f"{col_vals.min():.3f}-{col_vals.max():.3f}",
        "Median": round(float(col_vals.median()), 4),
    })

save_data_dictionary("breast_cancer", bc_rows)


Data dictionary -> d:\Projects\Agentic_AI\ml-explainability-agent\data\breast_cancer\data_dictionary.csv (30 features)


WindowsPath('d:/Projects/Agentic_AI/ml-explainability-agent/data/breast_cancer/data_dictionary.csv')

## Verify the saved models load through the app

This final step loads every saved `.joblib` back through the app's own
`ingestion.model_loader.ModelArtifactLoader`, confirming each one resolves to a supported
adapter and exposes normalized metadata (task type, feature names, framework). If this cell
succeeds, the models are ready to point the explainability agent at.


In [10]:
# Load every saved model back through the app's ingestion loader to confirm compatibility.
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ingestion.model_loader import ModelArtifactLoader

saved_models = sorted(MODELS_DIR.glob("*.joblib"))
summary_rows = []
for path in saved_models:
    loader = ModelArtifactLoader(str(path))
    meta = loader.extract_metadata()
    summary_rows.append({
        "file": path.name,
        "adapter": meta.get("adapter"),
        "model_type": meta.get("model_type"),
        "task_type": meta.get("task_type"),
        "framework": meta.get("framework", "-"),
        "n_features": meta.get("n_features"),
        "has_feature_names": bool(meta.get("feature_names")),
    })

pd.DataFrame(summary_rows)


,file,adapter,model_type,task_type,framework,n_features,has_feature_names
0,adult_income_random_forest.joblib,sklearn_tree,RandomForestClassifier,classification,scikit-learn,14,True
1,adult_income_xgboost.joblib,xgboost,XGBClassifier,classification,xgboost,14,True
2,breast_cancer_decision_tree.joblib,sklearn_tree,DecisionTreeClassifier,classification,scikit-learn,30,True
3,breast_cancer_extra_trees.joblib,sklearn_tree,ExtraTreesClassifier,classification,scikit-learn,30,True
4,california_housing_decision_tree.joblib,sklearn_tree,DecisionTreeRegressor,regression,scikit-learn,8,True
5,california_housing_random_forest.joblib,sklearn_tree,RandomForestRegressor,regression,scikit-learn,8,True
6,demo_iris_tree.joblib,sklearn_tree,DecisionTreeClassifier,classification,scikit-learn,4,True
7,test_model.joblib,generic,DecisionTreeClassifier,classification,-,0,False
